Se cargan las librerías para el análisis y la división posterior de los datos en entrenamiento y prueba.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split


## Carga y estructura

Se carga el conjunto transformado y se comprueba su tamaño, contenido inicial y tipos.

In [ ]:
df = pd.read_csv("sonrievalencia_citas_transformado.csv", sep=";")

print("Dataset cargado correctamente")
display(df.head())

print("Dimensiones del dataset:", df.shape)
print("Número de filas:", df.shape[0])
print("Número de columnas:", df.shape[1])
df.info()

## Estadísticas y valores ausentes

Se revisan los estadísticos descriptivos, los nulos por columna y su proporción.

In [ ]:
display(df.describe(include="all"))

print("Valores nulos por columna:")
display(df.isnull().sum())

print("Porcentaje de valores nulos:")
display((df.isnull().mean() * 100).round(2))

## Nulos escritos como texto y espacios sobrantes

Se comprueban marcadores de ausencia y espacios al inicio o al final en las columnas de texto.

In [ ]:
valores_nulos = [
    "-", "?", "N/A", "NA", "NULL",
    "NONE", "DESCONOCIDO", "NO DISPONIBLE"
]

print("BÚSQUEDA DE POSIBLES NULOS DISFRAZADOS")
for columna in df.select_dtypes(include="object").columns:
    valores = df[columna].astype("string").str.strip().str.upper()
    encontrados = valores.isin(valores_nulos).sum()

    if encontrados > 0:
        print(f"{columna}: {encontrados} posibles nulos disfrazados")

print("BÚSQUEDA DE ESPACIOS AL PRINCIPIO O FINAL")
for columna in df.select_dtypes(include="object").columns:
    espacios = df[columna].astype("string").str.contains(
        r"^\s|\s$", na=False
    ).sum()

    if espacios > 0:
        print(f"{columna}: {espacios} valores con espacios")

## Categorías y frecuencias

Se muestran las categorías observadas y la frecuencia de cada valor, incluidos los nulos.

In [ ]:
print("VALORES ÚNICOS DE CADA COLUMNA")
for columna in df.columns:
    print(f"\n--- {columna} ---")
    print(df[columna].unique())

print("FRECUENCIA DE VALORES POR COLUMNA")
for columna in df.columns:
    print(f"\n--- {columna} ---")
    print(df[columna].value_counts(dropna=False))

## Resumen

Se cierra la revisión con el tamaño, los nulos y los duplicados del conjunto.

In [ ]:
print("\n========== RESUMEN DEL DATASET ==========")
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
print("Total de valores nulos:", df.isnull().sum().sum())
print("Columnas con valores nulos:", df.isnull().gt(0).sum())
print("Duplicados:", df.duplicated().sum())
print("=========================================")

## Eliminar columnas constantes y duplicadas

Se trabaja sobre una copia para conservar `df` sin cambios. Primero se retiran columnas sin variación y luego columnas que repiten exactamente otra columna.

In [ ]:
df_reducido = df.copy()

print("DIMENSIONES INICIALES:", df_reducido.shape)

columnas_constantes = [
    columna
    for columna in df_reducido.columns
    if df_reducido[columna].nunique(dropna=False) <= 1
]

print("\nCOLUMNAS CONSTANTES:")
print(columnas_constantes)
df_reducido.drop(columns=columnas_constantes, inplace=True)

columnas_duplicadas = df_reducido.T[
    df_reducido.T.duplicated()
].index.tolist()

print("\nCOLUMNAS DUPLICADAS:")
print(columnas_duplicadas)
df_reducido.drop(columns=columnas_duplicadas, inplace=True)

## Detectar información redundante

Se comprueba si el código de clínica identifica exactamente una clínica y si el importe con IVA equivale al presupuesto más un 21 %.

In [ ]:
columnas_redundantes = []

if {"clinica", "cod_clinica"}.issubset(df_reducido.columns):
    equivalencia = (
        df_reducido.groupby("clinica")["cod_clinica"].nunique().max() == 1
        and df_reducido.groupby("cod_clinica")["clinica"].nunique().max() == 1
    )

    if equivalencia:
        columnas_redundantes.append("cod_clinica")

if {
    "importe_presupuesto_eur",
    "importe_con_iva_eur",
}.issubset(df_reducido.columns):
    base = df_reducido["importe_presupuesto_eur"]
    iva = df_reducido["importe_con_iva_eur"]
    validos = base.notna() & iva.notna()

    if validos.any():
        coincidencias = np.isclose(
            iva[validos],
            base[validos] * 1.21,
            atol=0.02,
        )
        porcentaje = coincidencias.mean()
        print("\nCoincidencia entre importes:", round(porcentaje * 100, 2), "%")

        if porcentaje >= 0.99:
            columnas_redundantes.append("importe_con_iva_eur")

print("\nCOLUMNAS REDUNDANTES:")
print(columnas_redundantes)
df_reducido.drop(columns=columnas_redundantes, inplace=True)

## Excluir identificadores

Los identificadores de cita y paciente no se conservan como variables para el análisis.

In [ ]:
identificadores = ["id_cita", "dni"]
identificadores_eliminados = [
    columna
    for columna in identificadores
    if columna in df_reducido.columns
]

print("\nIDENTIFICADORES EXCLUIDOS:")
print(identificadores_eliminados)
df_reducido.drop(columns=identificadores_eliminados, inplace=True)

## Revisar correlaciones altas

Se identifican pares de variables numéricas con correlación absoluta superior a 0,90. En cada pareja se elimina la segunda columna según el orden de la matriz.

In [ ]:
columnas_numericas = df_reducido.select_dtypes(
    include=np.number
).columns.tolist()
correlacion = df_reducido[columnas_numericas].corr().abs()

mascara = np.triu(np.ones(correlacion.shape), k=1).astype(bool)
correlaciones_altas = correlacion.where(mascara).stack()

print("\nCORRELACIONES SUPERIORES A 0.90:")
print(correlaciones_altas[correlaciones_altas > 0.90])

columnas_correlacionadas = set()
for (columna1, columna2), valor in correlaciones_altas.items():
    if valor > 0.90:
        columnas_correlacionadas.add(columna2)

print("\nCOLUMNAS ELIMINADAS POR CORRELACIÓN:")
print(sorted(columnas_correlacionadas))
df_reducido.drop(columns=list(columnas_correlacionadas), inplace=True)

## Resumen de reducción y balance

Se revisa qué columnas se han eliminado y cómo se distribuye `no_acude` antes de separar los datos.

In [ ]:
print("\n========== RESUMEN DE REDUCCIÓN ==========")
print("Dimensiones originales:", df.shape)
print("Dimensiones reducidas:", df_reducido.shape)
print("Columnas constantes:", columnas_constantes)
print("Columnas duplicadas:", columnas_duplicadas)
print("Columnas redundantes:", columnas_redundantes)
print("Identificadores excluidos:", identificadores_eliminados)
print("Columnas correlacionadas:", sorted(columnas_correlacionadas))
print("\nColumnas finales:")
print(df_reducido.columns.tolist())

print("\n========== DESBALANCEO DE CLASES ==========")
objetivo = "no_acude"

if objetivo not in df_reducido.columns:
    raise ValueError("No se encuentra la columna no_acude")

frecuencias = df_reducido[objetivo].value_counts()
print("\nFrecuencia de cada clase:")
print(frecuencias)
print("\nPorcentajes:")
print((df_reducido[objetivo].value_counts(normalize=True) * 100).round(2))

diferencia = frecuencias.max() / frecuencias.min()
print("\nProporción entre clases:", round(diferencia, 2))

if diferencia > 1.5:
    print("DESBALANCEO DETECTADO")
else:
    print("No se detecta un desbalanceo importante")

## Separar entrenamiento y prueba

La división estratificada mantiene la proporción de las clases y reserva el 20 % de los registros para prueba.

In [ ]:
X = df_reducido.drop(columns=[objetivo])
y = df_reducido[objetivo]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

## Equilibrar el conjunto de entrenamiento

La función sobremuestrea la clase minoritaria mediante repetición. Se usa solo con entrenamiento; los datos de prueba quedan sin modificar.

In [ ]:
def equilibrar_entrenamiento(X_train, y_train, objetivo, random_state=42):
    datos_train = X_train.copy()
    datos_train[objetivo] = y_train

    mayoritaria = y_train.value_counts().idxmax()
    minoritaria = y_train.value_counts().idxmin()

    datos_mayoritaria = datos_train[datos_train[objetivo] == mayoritaria]
    datos_minoritaria = datos_train[datos_train[objetivo] == minoritaria]

    datos_minoritaria_equilibrados = datos_minoritaria.sample(
        n=len(datos_mayoritaria),
        replace=True,
        random_state=random_state,
    )

    datos_equilibrados = pd.concat([
        datos_mayoritaria,
        datos_minoritaria_equilibrados,
    ])
    datos_equilibrados = datos_equilibrados.sample(
        frac=1,
        random_state=random_state,
    ).reset_index(drop=True)

    X_equilibrado = datos_equilibrados.drop(columns=[objetivo])
    y_equilibrado = datos_equilibrados[objetivo]
    return X_equilibrado, y_equilibrado


print("\n========== EQUILIBRADO DE CLASES ==========")
print("Antes del equilibrado:")
print(y_train.value_counts())

X_train_equilibrado, y_train_equilibrado = equilibrar_entrenamiento(
    X_train,
    y_train,
    objetivo,
)

## Comprobar el reparto final

Se comparan las clases del entrenamiento antes y después del sobremuestreo, y se confirma que el conjunto de prueba sigue intacto.

In [ ]:
print("\nANTES DEL EQUILIBRADO:")
print(y_train.value_counts())

print("\nDESPUÉS DEL EQUILIBRADO:")
print(y_train_equilibrado.value_counts())

print("\nDATOS DE PRUEBA (SIN MODIFICAR):")
print(y_test.value_counts())

print("\n========== RESULTADO FINAL ==========")
print("Filas de entrenamiento:", len(X_train_equilibrado))
print("Filas de prueba:", len(X_test))
print("Columnas finales:", X_train_equilibrado.shape[1])
print("\n¡REDUCCIÓN Y EQUILIBRADO COMPLETADOS!")

## Validar y preparar la vista final

La función retira campos que no se usarán como variables de entrada y comprueba que la vista tenga únicamente datos numéricos, sin nulos ni identificadores.


In [ ]:
def preparar_vista_final(df_reducido):
    df_final = df_reducido.copy()
    columnas_eliminar = [
        "clinica",
        "tratamiento",
        "seguro",
        "rango_edad",
        "fecha_reserva",
        "fecha_cita",
        "hora_cita",
        "notas_recepcion",
    ]

    df_final.drop(
        columns=[col for col in columnas_eliminar if col in df_final.columns],
        inplace=True,
    )

    columnas_no_numericas = df_final.select_dtypes(
        exclude=np.number
    ).columns.tolist()
    if columnas_no_numericas:
        raise ValueError(f"Hay columnas no numéricas: {columnas_no_numericas}")

    if df_final.isnull().sum().sum() > 0:
        raise ValueError("El dataset todavía contiene valores nulos.")

    assert "no_acude" in df_final.columns
    assert "dni" not in df_final.columns
    assert "id_cita" not in df_final.columns
    return df_final


## Guardar la vista definitiva

Se aplica la preparación y se exporta el CSV que consumirán los pasos posteriores.


In [ ]:
df_final = preparar_vista_final(df_reducido)
df_final.to_csv(
    "sonrievalencia_citas_final.csv",
    sep=";",
    index=False,
    encoding="utf-8",
)

print("CSV FINAL GENERADO CORRECTAMENTE")
print("Dimensiones:", df_final.shape)
print("Valores nulos:", df_final.isnull().sum().sum())
print(
    "Columnas no numéricas:",
    df_final.select_dtypes(exclude=np.number).columns.tolist(),
)
print("Variable objetivo:", "no_acude")
print("Distribución de clases:")
print(df_final["no_acude"].value_counts())
display(df_final.head())

Estas comprobaciones resumen el tipo de cada columna, las columnas no numéricas, los valores ausentes y la distribución de la variable objetivo en el archivo final.


In [6]:
print("TIPOS DE DATOS:")
display(df_final.dtypes)

print("\nCOLUMNAS NO NUMÉRICAS:")
print(df_final.select_dtypes(exclude=np.number).columns.tolist())

print("\nVALORES NULOS:")
print(df_final.isnull().sum())

print("\nDISTRIBUCIÓN DEL OBJETIVO:")
print(df_final["no_acude"].value_counts(dropna=False))

TIPOS DE DATOS:


edad                          float64
urgencia                        int64
primera_visita                  int64
visitas_previas                 int64
recordatorio_sms                int64
importe_presupuesto_eur       float64
no_acude                        int64
dias_antelacion                 int64
dia_semana_cita                 int64
es_tarde                        int64
miedo_dentista                  int64
tiene_nota                      int64
clinica_Benimaclet              int64
clinica_Campanar                int64
clinica_Ruzafa                  int64
seguro_Mutua                    int64
seguro_Privado                  int64
seguro_Sin seguro               int64
tratamiento_Blanqueamiento      int64
tratamiento_Carillas            int64
tratamiento_Cirugía             int64
tratamiento_Corona              int64
tratamiento_Empaste             int64
tratamiento_Endodoncia          int64
tratamiento_Extracción          int64
tratamiento_Férula              int64
tratamiento_


COLUMNAS NO NUMÉRICAS:
[]

VALORES NULOS:
edad                          0
urgencia                      0
primera_visita                0
visitas_previas               0
recordatorio_sms              0
importe_presupuesto_eur       0
no_acude                      0
dias_antelacion               0
dia_semana_cita               0
es_tarde                      0
miedo_dentista                0
tiene_nota                    0
clinica_Benimaclet            0
clinica_Campanar              0
clinica_Ruzafa                0
seguro_Mutua                  0
seguro_Privado                0
seguro_Sin seguro             0
tratamiento_Blanqueamiento    0
tratamiento_Carillas          0
tratamiento_Cirugía           0
tratamiento_Corona            0
tratamiento_Empaste           0
tratamiento_Endodoncia        0
tratamiento_Extracción        0
tratamiento_Férula            0
tratamiento_Implante          0
tratamiento_Limpieza          0
tratamiento_Ortodoncia        0
tratamiento_Periodoncia      

La suma de las columnas one-hot de `rango_edad` muestra cuántos registros pertenecen a cada tramo y permite contrastar ese total con el número de filas.


In [7]:
print(df_final[
    ['rango_edad_Menor',
     'rango_edad_Joven',
     'rango_edad_Adulto',
     'rango_edad_Senior']
].sum())

print("Total de registros:", len(df_final))

rango_edad_Menor      181
rango_edad_Joven      593
rango_edad_Adulto    1388
rango_edad_Senior     238
dtype: int64
Total de registros: 2400
